# Model Comparison & Explainability

Compare baseline classifiers and inspect feature importance.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

df = pd.read_csv("../data/raw/asd_screening_dataset.csv")
features = [c for c in df.columns if c not in ["ASD_Status","Child_ID"]]
X, y = df[features], df["ASD_Status"]
cat = ["Gender"]; num = [c for c in features if c not in cat]
pre = ColumnTransformer([
    ("num", StandardScaler(), num),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat)
])
X_train, X_test, y_train, y_test = train_test_split(
    X,y,test_size=.2,random_state=42,stratify=y
)


In [ ]:
candidates = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=42, class_weight="balanced")
}
rows=[]
for name, estimator in candidates.items():
    pipe = Pipeline([("preprocessor", pre), ("model", estimator)])
    pipe.fit(X_train,y_train)
    p=pipe.predict(X_test)
    prob=pipe.predict_proba(X_test)[:,1]
    rows.append({
        "model":name,
        "accuracy":accuracy_score(y_test,p),
        "f1":f1_score(y_test,p),
        "roc_auc":roc_auc_score(y_test,prob)
    })
pd.DataFrame(rows)


## Explainability

Use permutation importance, feature importance, and example-level explanations to understand model behavior. Avoid interpreting feature importance as causal evidence.